# Feature engineering (Dataset B)
Week 6: add features to the best tree model from 04 and show which ones help. CRISP-DM Data Preparation, task 3.3 Construct data: every derived attribute below says why it was made.

Deliverable (week-by-week doc): a table comparing old vs new feature sets, including the school district layer.

Dataset A is unchanged. Extra raw columns are joined here on `ListingId`, so the team's shared CSVs and 03 stay as they are. The feature functions live in `idx_pipeline.py`.

#### Setup

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline

from idx_pipeline import (CAT_COLS, DATA_DIR, SEED, TARGET, DatasetAImputer, add_basic_features, add_comps,
                          add_extras, add_fee_features, add_geo_features, add_market_trend, build_comp_pool,
                          build_preprocessor, error_by, fill_coords, load_extras, load_split, metrics)

print("sklearn", sklearn.__version__)
np.random.seed(SEED)
MODEL_DIR = os.path.join(DATA_DIR, "models")

#### Test design (written before training)
- **Split**: same as 03 and 04. Train = Jan 2024 – Feb 2026, validation = Mar 2026. Test is not loaded, and the raw April 2026 file isn't read.
- **Test model**: `rf_leaf5_log` from 04 (200 trees, `min_samples_leaf=5`, log target). It scored about the same as the full-depth forest with a much smaller overfit gap, and it's 1/6 the size.
- **Method**: add one feature group at a time, then the groups that helped together, then everything.
- **Noise check**: a paired bootstrap of validation homes gives a 95% interval on the MdAPE change. A group counts as helping only if the whole interval is below 0.
- **Leakage**: comps and market features use only sales from train that closed before the home did (handbook: comps are a fit-on-train transform).

#### Step 1: Load and build every feature

In [ ]:
raw = {s: load_split(s) for s in ["train", "val"]}
extras = load_extras()  # raw columns Dataset A leaves out; stops at Mar 2026
imputer = DatasetAImputer().fit(raw["train"])
coords = {s: fill_coords(raw[s], imputer) for s in raw}  # filled lat/long, for distances and neighbors

pool = build_comp_pool(raw["train"], *coords["train"])  # TRAIN ONLY
kmeans = MiniBatchKMeans(n_clusters=100, n_init=3, random_state=SEED).fit(np.column_stack(coords["train"]))  # TRAIN ONLY


def engineer(df, lat, lon):
    out = add_fee_features(add_basic_features(add_extras(df, extras)))
    out = add_geo_features(out, lat, lon)
    out["geo_cluster"] = kmeans.predict(np.column_stack([lat, lon])).astype(str)
    out = add_comps(out, lat, lon, pool)
    return add_market_trend(out, raw["train"])


start = time.time()
data = {s: engineer(raw[s], *coords[s]) for s in raw}
y = {s: d[TARGET].to_numpy() for s, d in data.items()}
print(f"{time.time() - start:.0f}s", {s: d.shape for s, d in data.items()})

In [ ]:
description = pd.read_csv(os.path.join(DATA_DIR, "dataset_description.csv"))
features = description.loc[description["role"].isin(["model_feature", "missing_flag"]), "column"].tolist()
BASE_NUM = [c for c in features if c not in CAT_COLS]  # Dataset A, same as 04

# group: (numeric columns, categorical columns), with the idea behind each
GROUPS = {
    # Team / week-by-week doc: ratios, Rohan's "a yard" and "combine the columns", flooring as a finish-quality signal
    "columns": (["bed_bath_ratio", "sqft_per_bed", "baths_per_bed", "stories_filled", "yard_sqft", "lot_coverage",
                 "main_level_share", "has_wood", "has_stone", "has_tile", "has_carpet", "has_laminate", "has_vinyl",
                 "Flooring_missing", "in_subdivision", "flag_small_for_beds", "flag_many_baths"],
                ["MLSAreaMajor", "SubdivisionName"]),
    # On hold from 02 until today: HOA fee and whether it's known
    "hoa": (["fee_monthly", "HasHOA", "AssociationFee_missing"], []),
    # Location was the biggest signal in 04: diagonals, commute (Stephanie), coast, neighborhoods finer than ZIP
    "geo": (["rot_15", "rot_30", "rot_45", "rot_60", "km_la", "km_sf", "km_sj", "km_sd", "km_sac", "km_job_center",
             "km_coast"], ["geo_cluster"]),
    # STAND-IN for the required school district layer, until the spatial join (needs Rohan's shapefile)
    "school_raw": ([], ["HighSchoolDistrict"]),
    # What nearby homes sold for in the 6 months before; aimed at the weak top price band
    "comps": (["comp_ppsf_median", "comp_ppsf_mean", "comp_km_kth", "comp_count", "sales_within_1km",
               "comp_implied_value"], []),
    # month_sin/cos only capture season; these capture whether prices are rising or falling
    "market": (["market_zip_3m", "market_county_3m", "market_county_12m", "market_county_change"], []),
}
for name, (num, cat) in GROUPS.items():
    print(f"{name}: {len(num)} numeric, {len(cat)} categorical")

In [ ]:
rows, apes, keep_models = {}, {}, {}


def run(name, extra_num=(), extra_cat=(), kind="tree_te", cats=CAT_COLS, keep=False):
    num, cat = BASE_NUM + list(extra_num), list(cats) + list(extra_cat)
    cols = list(dict.fromkeys(num + CAT_COLS + cat))
    pipe = Pipeline([("pre", build_preprocessor(kind, num, cat)),
                     ("model", RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED))])
    model = TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)
    start = time.time()
    model.fit(data["train"][cols], y["train"])  # TRAIN ONLY
    val_pred = model.predict(data["val"][cols])
    va, tr = metrics(y["val"], val_pred), metrics(y["train"], model.predict(data["train"][cols]))
    rows[name] = {**{f"val_{k}": v for k, v in va.items()}, "train_MdAPE_%": tr["MdAPE_%"],
                  "gap_MdAPE": round(va["MdAPE_%"] - tr["MdAPE_%"], 2), "n_inputs": len(num) + len(cat),
                  "fit_s": round(time.time() - start)}
    apes[name] = np.abs(val_pred - y["val"]) / y["val"]
    if keep:
        keep_models[name] = (model, cols)
    print(name, rows[name]["val_MdAPE_%"], rows[name]["val_MAPE_%"])


def bootstrap_change(new, ref, n=1000):
    # paired: the same resampled homes for both models. Points of MdAPE; negative = new is better.
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(apes[ref]), (n, len(apes[ref])))
    diff = 100 * (np.median(apes[new][idx], axis=1) - np.median(apes[ref][idx], axis=1))
    return round(np.median(apes[new]) * 100 - np.median(apes[ref]) * 100, 2), *np.round(np.percentile(diff, [2.5, 97.5]), 2)

#### Step 2: How to encode City, county and ZIP
Integer codes (what 04 used) vs target encoding (team decision, cross-fitted so a home's encoding never sees its own price), and ZIP alone, since Rohan's VIF check found these location columns overlap.

In [ ]:
run("base_ordinal", kind="tree")
run("target_encoded")
run("target_encoded_zip_only", cats=["PostalCode"])
pd.DataFrame(rows).T

#### Step 3: Add one feature group at a time
Each group is added to `target_encoded`, so all the new categoricals can be target-encoded too.

In [ ]:
for name, (num, cat) in GROUPS.items():
    run(f"+{name}", num, cat)

change = {name: bootstrap_change(f"+{name}", "target_encoded") for name in GROUPS}
ablation = pd.DataFrame(change, index=["MdAPE_change", "ci_low", "ci_high"]).T
ablation["helps"] = ablation["ci_high"] < 0
ablation.sort_values("MdAPE_change")

#### Step 4: Combine the groups that helped, then everything

In [ ]:
helped = ablation.index[ablation["helps"]].tolist()
print("groups that helped:", helped)
kept_num = [c for g in helped for c in GROUPS[g][0]]
kept_cat = [c for g in helped for c in GROUPS[g][1]]
run("kept_groups", kept_num, kept_cat, keep=True)

run("all_groups", [c for g in GROUPS.values() for c in g[0]], [c for g in GROUPS.values() for c in g[1]])
pd.DataFrame({"kept_groups": bootstrap_change("kept_groups", "target_encoded"),
              "all_groups": bootstrap_change("all_groups", "target_encoded")},
             index=["MdAPE_change", "ci_low", "ci_high"]).T

#### Step 5: Old vs new feature sets (week 6 deliverable)
Validation only, same 11,412 homes on every row. `school_raw` is a stand-in: the required spatial join needs Rohan's shapefile.

In [ ]:
results = pd.DataFrame(rows).T
assert all(len(a) == len(data["val"]) for a in apes.values())  # every row scored on the same homes
results.to_csv(os.path.join(MODEL_DIR, "fe_metrics.csv"))
results.sort_values("val_MdAPE_%")

#### Step 6: Where it helped: price bands

In [ ]:
val = data["val"]
bands = pd.qcut(val[TARGET], 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])
by_band = {}
for name in ["base_ordinal", "kept_groups"]:
    pred_ape = pd.Series(apes[name], index=val.index)
    by_band[name] = pd.DataFrame({"MdAPE_%": (100 * pred_ape.groupby(bands, observed=True).median()).round(2),
                                  "MAPE_%": (100 * pred_ape.groupby(bands, observed=True).mean()).round(2)})
pd.concat(by_band, axis=1)

#### Step 7: What the best model relies on
Permutation importance on validation, same as 04: shuffle one input and measure how much MdAPE worsens. Overlapping features share credit, so read groups, not single columns.

In [ ]:
model, cols = keep_models["kept_groups"]
mdape = make_scorer(lambda yt, yp: np.median(np.abs(yp - yt) / yt), greater_is_better=False)
pi = permutation_importance(model, val[cols], y["val"], scoring=mdape, n_repeats=2, random_state=SEED)
importance = (pd.DataFrame({"MdAPE_increase_pts": 100 * pi.importances_mean}, index=cols)
                .sort_values("MdAPE_increase_pts", ascending=False).round(2))
importance.to_csv(os.path.join(MODEL_DIR, "fe_permutation_importance.csv"))
importance.head(20)